In [19]:
import os
from langchain_community.embeddings import HuggingFaceEmbeddings  # 本地模型，需先 pip install sentence-transformers
from langchain.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain.document_loaders import TextLoader
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain

LM_STUDIO_BASE = "http://127.0.0.1:1234/v1"
llm = ChatOpenAI(
    base_url=LM_STUDIO_BASE,
    api_key="lm-studio",  # 本地服务可不校验，任意非空即可
    temperature=0,
    model="local",        # 若 LM Studio 要求指定模型名，可改为你在 LM Studio 里加载的模型名
)
# 相对路径：无论从项目根还是 notebookes 目录运行都能找到 test.csv
_csv = os.path.join("notebookes", "test.csv") if os.path.exists(os.path.join("notebookes", "test.csv")) else "test.csv"
loader = TextLoader(_csv)
documents = loader.load()

RuntimeError: Error loading ./notebookes/test.csv

In [16]:
text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 0)
documents = text_splitter.split_documents(documents)
# 本地 embedding：首次运行会下载模型，之后离线可用。中文可改用 "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents, embeddings)
memory= ConversationBufferMemory(memory_key = "chat_history", return_messages = True)
qa = ConversationalRetrievalChain.from_llm(llm, vectorstore.as_retriever (), memory = memory)

In [17]:
query= "我该怎么阅读transformer模型？"
result= qa({"question": query})
print(result["answer"])

/var/folders/h3/7zc3322s3y1bwclms8c2rjwc0000gn/T/ipykernel_78587/2345166386.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result= qa({"question": query})


你可以按照以下步骤来阅读 Transformer 模型：

1. **先跑通**：从头执行到底，确认 loss 会降。确保模型能够正常训练并收敛。

2. **抓数据线**：理解数据处理流程，从句子开始，经过 `make_data`（补 padding）生成 `enc_inputs`、`dec_inputs`、`dec_outputs`（LongTensor），然后通过 `DataLoader` 批量化，再输入模型。

3. **看结构**：对照「代码结构图」，在 notebook 中找到 Transformer → Encoder → EncoderLayer → MultiHeadAttention → ScaledDotProductAttention 的结构，理解各模块之间的调用关系。

4. **看形状**：在关键处打印张量形状（如 `print(x.shape)`），对照「张量形状简图」确认维度变化，例如 `(B, L, d_model)` 等。

5. **抠细节**：深入理解核心组件，如 ScaledDotProductAttention 的 QK^T、mask、softmax、乘 V；Decoder 中的 dec_self_attn 与 dec_enc_attn 各自的计算过程。

**两种图各有用**：
- **代码结构图**：理清「谁调谁」。
- **矩阵/张量图**：理清「维度怎么变」。

按需画一种或两种图，不必一次画全。


In [18]:
query= "数据流图有什么用？"
result= qa({"question": query})
print(result["answer"])

数据流图用于可视化和理解数据在系统或模型中的流动过程，帮助开发者和研究人员清晰地看到输入如何被处理、中间结果如何传递，以及最终输出是如何生成的。在 Transformer 模型中，数据流图可以：

1. **展示数据处理顺序**：例如，从输入句子 → 嵌入 → 位置编码 → 编码器/解码器层 → 输出 logits 的完整路径。
2. **帮助定位问题**：当出现维度不对、计算错误或逻辑错误时，通过数据流图可以快速定位数据在哪个环节被修改或转换。
3. **辅助调试与优化**：理解数据在每一步的形状和内容，有助于调试模型行为、调整超参数或优化结构。
4. **便于教学与沟通**：数据流图可以作为教学工具，帮助学生或团队成员理解模型的工作机制；也可以作为沟通工具，让非技术人员理解模型的运作方式。
5. **支持模块化设计**：通过数据流图可以清晰地看到每个模块（如编码器、解码器、注意力层）的输入输出关系，从而支持模块化开发和复用。

在 Transformer 的上下文中，数据流图特别有用，因为它能直观展示从输入句子到最终预测词的完整路径，包括嵌入、位置编码、多层注意力和前馈网络的处理过程，以及最终的投影输出。这有助于理解模型结构、调试问题和优化性能。
